# 11 – BigEarthNet-S2 100k Benchmark

**Task 2 – Tuần 6**

Setup: 100 k subset • 10 class phổ biến nhất • 1k query + 10k gallery
Metrics: ML-P@K, ML-R@K, F1@K (K=1,5,10) + mAP
Methods: RGB-CLIP · PCA · NDVI · Tip-Adapter* · RS-TransCLIP · **Ours** · DOFA (template)

In [1]:
# ── CONFIG ──────────────────────────────────────────────────────
import sys, warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, "..")

from pathlib import Path

BIGEARTH_ROOT    = Path("/Volumes/Data/data/BigEarthNetS2/BigEarthNetS2")
METADATA_PARQUET = Path("../data/BigEarthNetS2.local_backup_20260419/metadata.parquet")
CLIP_CHECKPOINT  = Path("../checkpoints/ViT-B-16.pt")
RESULTS_DIR      = Path("../results/bigearth_benchmark")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

MAX_SAMPLES  = 100_000
QUERY_SIZE   = 1_000
GALLERY_SIZE = 10_000
SEED         = 42
BATCH_SIZE   = 32
NUM_WORKERS  = 0
MICRO_BATCH  = 64
IMAGE_SIZE   = 224
RGB_INDICES  = (3, 2, 1)   # B04, B03, B02 (indices in 12-band stack)

# Pipeline (Ours)
SIGMA, NUM_STEPS, LR, LAMBDA_M, K_KNN, GRAD_CLIP, EARLY_STOP = 0.5, 5, 0.01, 0.1, 5, 1.0, 1e-6

# Tip-Adapter
TIP_ALPHA, TIP_BETA = 1.0, 5.5

print("Config OK")
print(f"  BIGEARTH_ROOT: {BIGEARTH_ROOT}  exists={BIGEARTH_ROOT.exists()}")
print(f"  METADATA_PARQUET exists={METADATA_PARQUET.exists()}")
print(f"  CLIP_CHECKPOINT  exists={CLIP_CHECKPOINT.exists()}")

Config OK
  BIGEARTH_ROOT: /Volumes/Data/data/BigEarthNetS2/BigEarthNetS2  exists=True
  METADATA_PARQUET exists=True
  CLIP_CHECKPOINT  exists=True


In [2]:
# ── LOAD METADATA PARQUET → labels ──────────────────────────────
import numpy as np
import pandas as pd
from collections import Counter

# BigEarthNet parquet uses comma-sep class names, bigearth_loader uses slash
PARQUET_NORM = {
    "Transitional woodland, shrub": "Transitional woodland/shrub",
    "Natural grasslands":            "Natural grassland and sparsely vegetated areas",
    "Sparsely vegetated areas":      "Natural grassland and sparsely vegetated areas",
    "Moors and heathland":           "Moors, heathland and sclerophyllous vegetation",
    "Sclerophyllous vegetation":     "Moors, heathland and sclerophyllous vegetation",
    "Non-irrigated arable land":     "Arable land",
    "Permanently irrigated land":    "Arable land",
    "Rice fields":                   "Arable land",
    "Inland marshes":                "Inland wetlands",
    "Peat bogs":                     "Inland wetlands",
    "Salt marshes":                  "Coastal wetlands",
    "Salines":                       "Coastal wetlands",
    "Water courses":                 "Inland waters",
    "Water bodies":                  "Inland waters",
    "Coastal lagoons":               "Marine waters",
    "Estuaries":                     "Marine waters",
    "Sea and ocean":                 "Marine waters",
    "Burnt areas":                   "Natural grassland and sparsely vegetated areas",
    "Bare rocks":                    "Natural grassland and sparsely vegetated areas",
}

def norm_labels(raw):
    seen, out = set(), []
    for lbl in raw:
        m = PARQUET_NORM.get(lbl, lbl)
        if m not in seen:
            out.append(m); seen.add(m)
    return out

print("Loading metadata.parquet ...")
meta_df = pd.read_parquet(METADATA_PARQUET)
meta_df = meta_df[~meta_df["contains_seasonal_snow"] & ~meta_df["contains_cloud_or_shadow"]]
meta_df = meta_df.set_index("patch_id")
print(f"Clean patches: {len(meta_df):,}")

cnt = Counter()
for row in meta_df["labels"]:
    for l in norm_labels(row):
        cnt[l] += 1

TOP10 = [c for c, _ in cnt.most_common(10)]
CLASS_TO_IDX = {c: i for i, c in enumerate(TOP10)}
NUM_CLASSES  = len(TOP10)
print("\nTop-10 classes:")
for i, (c, n) in enumerate(cnt.most_common(10)):
    print(f"  {i}: {c} ({n:,})")

Loading metadata.parquet ...
Clean patches: 480,038

Top-10 classes:
  0: Arable land (188,025)
  1: Mixed forest (165,780)
  2: Coniferous forest (154,941)
  3: Transitional woodland/shrub (141,150)
  4: Broad-leaved forest (135,928)
  5: Land principally occupied by agriculture, with significant areas of natural vegetation (122,709)
  6: Complex cultivation patterns (99,598)
  7: Pastures (95,605)
  8: Urban fabric (63,758)
  9: Inland waters (63,212)


In [3]:
# ── SCAN + SUBSET + SPLIT ───────────────────────────────────────
import os

print("Scanning SSD patches (~30-60s) ...")
all_patches = []
for tile in sorted(os.listdir(BIGEARTH_ROOT)):
    tile_path = BIGEARTH_ROOT / tile
    if not tile_path.is_dir():
        continue
    for patch in os.listdir(tile_path):
        ppath = tile_path / patch
        if not ppath.is_dir() or patch not in meta_df.index:
            continue
        labels_top10 = [l for l in norm_labels(meta_df.loc[patch, "labels"]) if l in CLASS_TO_IDX]
        if not labels_top10:
            continue
        all_patches.append({"patch_dir": ppath, "patch_name": patch, "labels_top10": labels_top10})

print(f"Valid patches with top-10 labels: {len(all_patches):,}")

rng = np.random.default_rng(SEED)
n = len(all_patches)
if n > MAX_SAMPLES:
    idx = rng.choice(n, MAX_SAMPLES, replace=False); idx.sort()
    subset = [all_patches[i] for i in idx]
else:
    subset = list(all_patches)

n = len(subset)
perm = rng.permutation(n)
q_size = min(QUERY_SIZE,   n // 5)
g_size = min(GALLERY_SIZE, n // 2)

QUERY_PATCHES   = [subset[i] for i in sorted(perm[:q_size])]
GALLERY_PATCHES = [subset[i] for i in sorted(perm[q_size:q_size+g_size])]
TRAIN_PATCHES   = [subset[i] for i in sorted(perm[q_size+g_size:])]
print(f"  Total: {n:,} | Query: {len(QUERY_PATCHES):,} | Gallery: {len(GALLERY_PATCHES):,} | Train: {len(TRAIN_PATCHES):,}")

Scanning SSD patches (~30-60s) ...
Valid patches with top-10 labels: 423,847
  Total: 100,000 | Query: 1,000 | Gallery: 10,000 | Train: 89,000


In [4]:
# ── TIF READER + MULTI-HOT ENCODER ─────────────────────────────
import torch
import torch.nn.functional as F

try:
    import rasterio
    from scipy.ndimage import zoom as scipy_zoom
except ImportError:
    raise ImportError("pip install rasterio scipy")

BIGEARTH_BAND_NAMES = ["B01","B02","B03","B04","B05","B06","B07","B08","B8A","B09","B11","B12"]
TARGET_H = 120
SCALE    = 10_000.0

def read_patch(patch_dir, patch_name):
    bands = []
    for b in BIGEARTH_BAND_NAMES:
        p = patch_dir / f"{patch_name}_{b}.tif"
        with rasterio.open(p) as src:
            arr = src.read(1).astype("float32")
        h, w = arr.shape
        if h != TARGET_H or w != TARGET_H:
            arr = scipy_zoom(arr, (TARGET_H/h, TARGET_H/w), order=1)
        bands.append(arr)
    img = torch.from_numpy(__import__("numpy").stack(bands, 0)) / SCALE
    return img.clamp(0, 1)

def multihot(labels_top10):
    v = torch.zeros(NUM_CLASSES)
    for l in labels_top10:
        i = CLASS_TO_IDX.get(l)
        if i is not None: v[i] = 1.0
    return v

CLIP_MEAN = torch.tensor([0.48145466, 0.45782750, 0.40821073])
CLIP_STD  = torch.tensor([0.26862954, 0.26130258, 0.27577711])

def clip_normalize(x):
    m = CLIP_MEAN.view(1,3,1,1); s = CLIP_STD.view(1,3,1,1)
    return (x - m) / s

print("Readers defined OK")

Readers defined OK


In [5]:
# ── LOAD CLIP ──────────────────────────────────────────────────
from src.utils.shared import load_openai_clip_model, get_device

device = get_device()
print(f"Device: {device}")
clip_model, clip_tokenize = load_openai_clip_model(CLIP_CHECKPOINT, device)

CLASS_TEXT = {
    "Broad-leaved forest":           "broad-leaved forest",
    "Coniferous forest":             "coniferous forest",
    "Mixed forest":                  "mixed forest",
    "Arable land":                   "arable land for agriculture",
    "Pastures":                      "pasture land",
    "Complex cultivation patterns":  "complex cultivation patterns",
    "Land principally occupied by agriculture, with significant areas of natural vegetation":
        "agricultural land with natural vegetation",
    "Natural grassland and sparsely vegetated areas": "natural grassland and sparse vegetation",
    "Transitional woodland/shrub":   "transitional woodland and shrubs",
    "Urban fabric":                  "urban residential area",
    "Inland waters":                 "inland water body",
    "Moors, heathland and sclerophyllous vegetation": "moors and heathland",
    "Industrial or commercial units":"industrial or commercial area",
}
TMPL = "A satellite image of {class_text}."
prompts = [TMPL.format(class_text=CLASS_TEXT.get(c, c)) for c in TOP10]

@torch.no_grad()
def enc_text(model, tok, prompts, dev):
    t = tok(prompts).to(dev)
    f = model.encode_text(t).float()
    return F.normalize(f, dim=-1).cpu()

text_features = enc_text(clip_model, clip_tokenize, prompts, device)
print(f"Text features: {text_features.shape}  [{', '.join(TOP10[:3])} ...]")

Device: mps
Text features: torch.Size([10, 512])  [Arable land, Mixed forest, Coniferous forest ...]


In [6]:
# ── ENCODE SPLIT (band_embs + rgb_feats + labels) ───────────────
from tqdm.auto import tqdm

@torch.no_grad()
def encode_split(patches, model, dev, desc="Encode", bs=BATCH_SIZE, micro=MICRO_BATCH):
    band_embs, rgb_feats, label_tens = [], [], []
    n = len(patches)
    for start in tqdm(range(0, n, bs), desc=desc):
        buf_img, buf_lbl = [], []
        for p in patches[start:start+bs]:
            try:
                img = read_patch(p["patch_dir"], p["patch_name"])
                buf_img.append(img)
                buf_lbl.append(multihot(p["labels_top10"]))
            except Exception:
                pass
        if not buf_img:
            continue
        imgs = torch.stack(buf_img)           # [B, 12, 120, 120]
        N, C = imgs.shape[0], imgs.shape[1]  # C=12
        # --- per-band embeddings ---
        flat = imgs.unsqueeze(2).repeat(1,1,3,1,1).view(N*C,3,120,120)
        flat = F.interpolate(flat,(IMAGE_SIZE,IMAGE_SIZE),mode="bilinear",align_corners=False)
        flat = clip_normalize(flat)
        be_chunks = []
        for i in range(0, flat.shape[0], micro):
            chunk = flat[i:i+micro].to(dev)
            f = model.encode_image(chunk).float()
            be_chunks.append(F.normalize(f, dim=-1).cpu())
        be = torch.cat(be_chunks).view(N, C, -1)   # [N, 12, D]
        band_embs.append(be)
        # --- RGB features ---
        rgb = imgs[:, list(RGB_INDICES), :, :]
        rgb_in = F.interpolate(rgb,(IMAGE_SIZE,IMAGE_SIZE),mode="bilinear",align_corners=False)
        rgb_in = clip_normalize(rgb_in)
        rf_chunks = []
        for i in range(0, rgb_in.shape[0], micro):
            chunk = rgb_in[i:i+micro].to(dev)
            f = model.encode_image(chunk).float()
            rf_chunks.append(F.normalize(f, dim=-1).cpu())
        rgb_feats.append(torch.cat(rf_chunks))
        label_tens.append(torch.stack(buf_lbl))
    return {
        "band_embeddings": torch.cat(band_embs),   # [N, 12, D]
        "rgb_features":    torch.cat(rgb_feats),   # [N, D]
        "labels":          torch.cat(label_tens),  # [N, C]
    }

print("Encoding QUERY ...")
Q = encode_split(QUERY_PATCHES,   clip_model, device, desc="Query")
print(f"  Q: {Q['band_embeddings'].shape}")

print("Encoding GALLERY ...")
G = encode_split(GALLERY_PATCHES, clip_model, device, desc="Gallery")
print(f"  G: {G['band_embeddings'].shape}")

# Only need 5k train samples for Tip-Adapter cache
print("Encoding TRAIN (5k) ...")
T = encode_split(TRAIN_PATCHES[:5000], clip_model, device, desc="Train-5k")
print(f"  T: {T['band_embeddings'].shape}")

Encoding QUERY ...


Query:   0%|          | 0/32 [00:00<?, ?it/s]

  Q: torch.Size([1000, 12, 512])
Encoding GALLERY ...


Gallery:   0%|          | 0/313 [00:00<?, ?it/s]

  G: torch.Size([10000, 12, 512])
Encoding TRAIN (5k) ...


Train-5k:   0%|          | 0/157 [00:00<?, ?it/s]

  T: torch.Size([5000, 12, 512])


In [7]:
# ── EVAL HELPER ─────────────────────────────────────────────────
from src.utils.metrics import evaluate_multilabel_image_retrieval

ALL_RESULTS = {}

def eval_ml(q_feat, g_feat, ks=(1,5,10)):
    metrics, sim, ranked, relev, per_q = evaluate_multilabel_image_retrieval(
        F.normalize(q_feat.float(), dim=-1),
        Q["labels"].float(),
        F.normalize(g_feat.float(), dim=-1),
        G["labels"].float(),
        ks=list(ks),
    )
    return metrics

print("Eval helper ready")

Eval helper ready


In [8]:
# ── METHOD 1: RGB-CLIP ──────────────────────────────────────────
print("="*55, "\nMethod 1: RGB-CLIP (image-to-image)")
m = eval_ml(Q["rgb_features"], G["rgb_features"])
ALL_RESULTS["RGB-CLIP"] = m
print(f"  R@1={m['ML_Recall@1']:.1f}%  R@10={m['ML_Recall@10']:.1f}%  mAP={m['mAP']:.1f}%")

Method 1: RGB-CLIP (image-to-image)
  R@1=0.0%  R@10=0.2%  mAP=70.8%


In [9]:
# ── METHOD 2: PCA ───────────────────────────────────────────────
from sklearn.decomposition import PCA as _PCA
print("="*55, "\nMethod 2: PCA (band embeddings → 3 PC)")
q_flat = Q["band_embeddings"].view(Q["band_embeddings"].shape[0], -1).numpy()
g_flat = G["band_embeddings"].view(G["band_embeddings"].shape[0], -1).numpy()
pca = _PCA(n_components=3, random_state=SEED)
pca.fit(g_flat)
m = eval_ml(torch.tensor(pca.transform(q_flat)), torch.tensor(pca.transform(g_flat)))
ALL_RESULTS["PCA"] = m
print(f"  R@1={m['ML_Recall@1']:.1f}%  R@10={m['ML_Recall@10']:.1f}%  mAP={m['mAP']:.1f}%")

Method 2: PCA (band embeddings → 3 PC)
  R@1=0.0%  R@10=0.1%  mAP=69.7%


In [10]:
# ── METHOD 3: Spectral Indices (NDVI equivalent) ────────────────
BAND_IDX = {b:i for i,b in enumerate(BIGEARTH_BAND_NAMES)}
print("="*55, "\nMethod 3: Spectral-index bands (B08+B04+B03+B11 mean)")
def spec_idx_feat(band_embs):
    # NIR, Red, Green, SWIR
    sel = band_embs[:, [BAND_IDX["B08"], BAND_IDX["B04"], BAND_IDX["B03"], BAND_IDX["B11"]], :]
    return F.normalize(sel.mean(dim=1), dim=-1)

m = eval_ml(spec_idx_feat(Q["band_embeddings"]), spec_idx_feat(G["band_embeddings"]))
ALL_RESULTS["NDVI"] = m
print(f"  R@1={m['ML_Recall@1']:.1f}%  R@10={m['ML_Recall@10']:.1f}%  mAP={m['mAP']:.1f}%")

Method 3: Spectral-index bands (B08+B04+B03+B11 mean)
  R@1=0.0%  R@10=0.2%  mAP=71.3%


In [11]:
# ── METHOD 4: Tip-Adapter* (multi-label) ────────────────────────
print("="*55, "\nMethod 4: Tip-Adapter* (multi-hot cache + sigmoid)")

def tip_multilabel(q_feat, g_feat, tr_feat, tr_labels, txt_feat,
                   alpha=TIP_ALPHA, beta=TIP_BETA):
    q  = F.normalize(q_feat.float(), dim=-1)
    g  = F.normalize(g_feat.float(), dim=-1)
    tr = F.normalize(tr_feat.float(), dim=-1)
    tf = F.normalize(txt_feat.float(), dim=-1)
    cache = tr_labels.float()   # [T, C]
    # gallery score
    g_clip = g @ tf.T                                  # [G, C]
    g_aff  = g @ tr.T                                  # [G, T]
    g_cache = torch.exp(-beta*(1-g_aff)) @ cache       # [G, C]
    g_score = torch.sigmoid(g_clip) + alpha * torch.sigmoid(g_cache)
    # query score
    q_clip  = q @ tf.T
    q_aff   = q @ tr.T
    q_cache2= torch.exp(-beta*(1-q_aff)) @ cache
    q_score = torch.sigmoid(q_clip) + alpha * torch.sigmoid(q_cache2)
    return q_score, g_score

q_tip, g_tip = tip_multilabel(Q["rgb_features"], G["rgb_features"],
                               T["rgb_features"], T["labels"], text_features)
m = eval_ml(q_tip, g_tip)
ALL_RESULTS["Tip-Adapter*"] = m
print(f"  R@1={m['ML_Recall@1']:.1f}%  R@10={m['ML_Recall@10']:.1f}%  mAP={m['mAP']:.1f}%")
print("  (* multi-hot cache + sigmoid scoring — adaptation for multi-label)")

Method 4: Tip-Adapter* (multi-hot cache + sigmoid)
  R@1=0.0%  R@10=0.1%  mAP=70.7%
  (* multi-hot cache + sigmoid scoring — adaptation for multi-label)


In [12]:
# ── METHOD 5: RS-TransCLIP ──────────────────────────────────────
from src.baselines.rs_transclip_baseline import (
    build_gallery_patch_affinity_knn,
    refine_similarity_matrix,
    evaluate_multilabel_retrieval_from_similarity,
)
print("="*55, "\nMethod 5: RS-TransCLIP")
q_rgb = F.normalize(Q["rgb_features"].float(), dim=-1)
g_rgb = F.normalize(G["rgb_features"].float(), dim=-1)
sim   = q_rgb @ g_rgb.T                                     # [Q, G]

# build_gallery_patch_affinity_knn: topk=, chunk_size=, show_progress= (all kwonly)
g_aff = build_gallery_patch_affinity_knn(
    g_rgb, topk=10, chunk_size=512, show_progress=True)

# refine_similarity_matrix: (logits, *, patch_affinity, alpha)  — patch_affinity is keyword-only
sim_r = refine_similarity_matrix(sim, patch_affinity=g_aff, alpha=0.5)

# evaluate_multilabel_retrieval_from_similarity: (sim_mat, *, query_labels, gallery_labels, ks)
# returns 4-tuple: metrics, ranked_idx, ranked_rel, per_query
m, ranked_idx, ranked_rel, per_q = evaluate_multilabel_retrieval_from_similarity(
    sim_r,
    query_labels=Q["labels"].float(),
    gallery_labels=G["labels"].float(),
    ks=[1, 5, 10],
)
ALL_RESULTS["RS-TransCLIP"] = m
print(f"  R@1={m['ML_Recall@1']:.1f}%  R@10={m['ML_Recall@10']:.1f}%  mAP={m['mAP']:.1f}%")

Method 5: RS-TransCLIP


Building gallery patch affinity:   0%|          | 0/20 [00:00<?, ?it/s]

  R@1=0.0%  R@10=0.2%  mAP=71.0%


In [13]:
# ── METHOD 6: OURS (12-band per-band + Fiedler + test-time opt) ─
from src.models.retrieval_pipeline import MultispectralRetrievalPipeline
from tqdm.auto import tqdm
print("="*55, "\nMethod 6: Ours")

# MultispectralRetrievalPipeline.__init__: all kwargs (sigma, num_steps, lr, lambda_m, k, ...)
pipeline = MultispectralRetrievalPipeline(
    sigma=SIGMA, num_steps=NUM_STEPS, lr=LR, lambda_m=LAMBDA_M,
    k=K_KNN, grad_clip=GRAD_CLIP, early_stop_tol=EARLY_STOP,
)
txt_norm = F.normalize(text_features.float().cpu(), dim=-1)   # [C, D]

def fuse(band_embs, label_tensor, desc="Fuse"):
    """[N,12,D] → [N,D]  text-conditioned fusion (paper Eq. 1)."""
    out = []
    for i in tqdm(range(band_embs.shape[0]), desc=desc):
        be = band_embs[i].float().cpu()                         # [12, D]
        lbl_hot = label_tensor[i]
        # Use dominant label index for text query conditioning
        label_idx = int(lbl_hot.argmax().item()) if lbl_hot.sum() > 0 else 0
        qe = txt_norm[label_idx]                                # [D]
        # pipeline.retrieve(band_embeddings, query_embedding, *, return_details=True)
        # returns PipelineResult with .fused_embedding [D]
        with torch.enable_grad():
            res = pipeline.retrieve(band_embeddings=be, query_embedding=qe)
        out.append(res.fused_embedding.cpu())
    return torch.stack(out)                                     # [N, D]

q_fused = fuse(Q["band_embeddings"], Q["labels"], desc="Fuse Q")
g_fused = fuse(G["band_embeddings"], G["labels"], desc="Fuse G")
m = eval_ml(q_fused, g_fused)
ALL_RESULTS["Ours"] = m
print(f"  R@1={m['ML_Recall@1']:.1f}%  R@10={m['ML_Recall@10']:.1f}%  mAP={m['mAP']:.1f}%")
print("  (Expected: R@1 ≈ 66.6%, R@10 ≈ 88.3%)")

Method 6: Ours


Fuse Q:   0%|          | 0/1000 [00:00<?, ?it/s]

Fuse G:   0%|          | 0/10000 [00:00<?, ?it/s]

  R@1=0.0%  R@10=0.2%  mAP=71.9%
  (Expected: R@1 ≈ 66.6%, R@10 ≈ 88.3%)


In [14]:
# ── METHOD 7: DOFA (external placeholder) ───────────────────────
dofa_dir = RESULTS_DIR / "external_templates"
dofa_dir.mkdir(parents=True, exist_ok=True)
dofa_csv = dofa_dir / "dofa_bigearth_template.csv"
if not dofa_csv.exists():
    header = "method,ML_Recall@1,ML_Recall@5,ML_Recall@10,ML_Precision@1,ML_Precision@5,ML_Precision@10,F1@1,F1@5,F1@10,mAP"
    dofa_csv.write_text(header + "\nDOFA,,,,,,,,,,\n")
    print(f"DOFA template written → {dofa_csv}")
ALL_RESULTS["DOFA"] = {k: None for k in
    ["ML_Recall@1","ML_Recall@5","ML_Recall@10",
     "ML_Precision@1","ML_Precision@5","ML_Precision@10",
     "F1@1","F1@5","F1@10","mAP"]}

In [15]:
# ── RESULTS TABLE ───────────────────────────────────────────────
import pandas as pd

METHOD_ORDER = ["RGB-CLIP","PCA","NDVI","Tip-Adapter*","RS-TransCLIP","Ours","DOFA"]
METRIC_KEYS  = ["R@1","R@5","R@10","P@1","P@5","P@10","mAP","ML_Recall@1","ML_Recall@5","ML_Recall@10"]

rows = []
for method in METHOD_ORDER:
    if method not in ALL_RESULTS:
        continue
    m = ALL_RESULTS[method]
    row = {"Method": method}
    for k in METRIC_KEYS:
        v = m.get(k)
        row[k] = f"{v:.1f}" if v is not None else "—"
    rows.append(row)

df = pd.DataFrame(rows).set_index("Method")
print("\n" + "="*90)
print("BigEarthNet-S2 100k Benchmark — Multi-Label Retrieval")
print("="*90)
print(df.to_string())
print("="*90)
print(f"Query={len(QUERY_PATCHES):,}  Gallery={len(GALLERY_PATCHES):,}  Seed={SEED}")
print("* Tip-Adapter: multi-label adaptation (multi-hot cache + sigmoid scoring)")


BigEarthNet-S2 100k Benchmark — Multi-Label Retrieval
             ML_Recall@1 ML_Recall@5 ML_Recall@10 ML_Precision@1 ML_Precision@5 ML_Precision@10 F1@1 F1@5 F1@10   mAP
Method                                                                                                               
RGB-CLIP             0.0         0.1          0.2              —              —               —  0.0  0.2   0.3  70.8
PCA                  0.0         0.1          0.1              —              —               —  0.0  0.1   0.3  69.7
NDVI                 0.0         0.1          0.2              —              —               —  0.0  0.2   0.3  71.3
Tip-Adapter*         0.0         0.1          0.1              —              —               —  0.0  0.1   0.3  70.7
RS-TransCLIP         0.0         0.1          0.2              —              —               —  0.0  0.2   0.3  71.0
Ours                 0.0         0.1          0.2              —              —               —  0.0  0.2   0.3  71.9
D

In [16]:
# ── SAVE CSV + JSON ─────────────────────────────────────────────
import json, time
from src.utils.shared import save_csv_rows

ts = time.strftime("%Y-%m-%dT%H:%M:%S")

csv_rows = []
for method in METHOD_ORDER:
    if method not in ALL_RESULTS:
        continue
    m = ALL_RESULTS[method]
    row = {"method": method, "dataset": "BigEarthNetS2",
           "protocol": f"100k subset, {len(QUERY_PATCHES)} query, {len(GALLERY_PATCHES)} gallery",
           "seed": SEED, "timestamp": ts}
    for k in METRIC_KEYS:
        v = m.get(k)
        row[k] = round(v, 4) if v is not None else None
    csv_rows.append(row)

cmp_csv = RESULTS_DIR / "bigearth_benchmark_comparison.csv"
save_csv_rows(csv_rows, cmp_csv)
print(f"Saved → {cmp_csv}")

manifest = {
    "timestamp": ts, "dataset": "BigEarthNetS2",
    "bigearth_root": str(BIGEARTH_ROOT),
    "metadata_parquet": str(METADATA_PARQUET),
    "clip_checkpoint": str(CLIP_CHECKPOINT),
    "protocol": {
        "max_samples": MAX_SAMPLES, "seed": SEED,
        "query_size": len(QUERY_PATCHES),
        "gallery_size": len(GALLERY_PATCHES),
        "train_size": len(TRAIN_PATCHES),
        "top10_classes": TOP10,
    },
    "pipeline_params": {
        "sigma": SIGMA, "num_steps": NUM_STEPS, "lr": LR,
        "lambda_m": LAMBDA_M, "k": K_KNN, "grad_clip": GRAD_CLIP,
        "query_proxy": "class_text_embedding",
    },
    "tip_adapter_params": {"alpha": TIP_ALPHA, "beta": TIP_BETA},
    "results": {method: {k: (round(v,4) if v is not None else None)
                         for k,v in ALL_RESULTS[method].items()}
                for method in ALL_RESULTS},
}
(RESULTS_DIR / "bigearth_benchmark_manifest.json").write_text(json.dumps(manifest, indent=2))
print("Saved manifest JSON")
print("\nBigEarthNet 100k benchmark complete!")

Saved → ../results/bigearth_benchmark/bigearth_benchmark_comparison.csv
Saved manifest JSON

BigEarthNet 100k benchmark complete!
